# Migrate MuJoCo to Newton — Keep the Task, Change the API

You have now built and stepped a complete Newton model. Bring back the Article 2 task: pick up a red cube and stack it on a blue cube, first with the **SO-101**, then with the **Seeed reBot DevArm**.

You are not replacing the controller, redefining success or making a new robot asset. You are moving scene construction and dynamics behind Newton's `ModelBuilder`, `State`, `Control`, `Contacts` and `SolverMuJoCo`. That gives you a small, reviewable migration before the later notebook adds cloth, cable and explicit solver coupling.

The external files contain intentional exercises. This notebook runs the finished references by default, so a clean checkout is readable and runnable; when your implementation is ready, switch to student mode and verify it without overwriting either version.

## Outline

- Select the Newton kernel and preserve your exercise files.
- Check the incomplete scene's specific baseline error, rather than accepting any failed process.
- Complete `newton_scene.py`, **Steps 0–5**: import, builder, robot, targets, props, finalize.
- Complete `so101_newton.py`, **Steps 6–9**: buffers, solver, substeps and optional CUDA graph.
- Verify the CPU MuJoCo baseline and Newton migration for **both robots**.
- Inspect the viewer and CUDA contact paths separately, without claiming unrun checks passed.

## What you will learn

By the end of this notebook, you will be able to:

- Reuse an MJCF robot inside a Newton-authored scene while preserving task geometry and mass.
- Map actuator names to coordinate, velocity-DOF and position-target indices safely.
- Keep the controller clock aligned with physics substeps.
- Distinguish native MuJoCo CPU from MuJoCo Warp, and contact detection from integration.
- Assert a physical stack instead of treating a successful exit code as sufficient.

## Setup: one interpreter, an explicit mode

From `tutorials/newton`, use the pinned Newton environment:

```bash
uv venv --python 3.12 .venv
uv pip install --python .venv/bin/python --require-hashes -r notebooks/newton/requirements.lock.txt
# usd-core 26.3 on every platform: retain this startup environment for the later USD lessons.
PXR_WORK_THREAD_LIMIT=1 .venv/bin/jupyter lab --notebook-dir notebooks
```

Select that interpreter as your notebook kernel and start with **`newton/part3` as the working directory**. All subprocesses below use `sys.executable`, an explicit `cwd` and `subprocess.run(check=True)`: a different shell `python` cannot silently run the wrong environment, and a failed task stops the notebook.

With `usd-core` 26.3 on every platform, launch Python/Jupyter with `PXR_WORK_THREAD_LIMIT=1` before the later official-USD imports. Native parallel-parser aborts were observed on macOS ARM and Linux; no Windows reproduction is claimed. Restart an existing server/kernel rather than assigning it after `pxr` loads. This is an observed parser-concurrency mitigation, not a physics tuning parameter; this migration notebook deliberately keeps the MJCF asset.

Set `REFERENCE = True` to read and run the finished files directly. Keep your own changes in `newton_scene.py` and `so101_newton.py`; **no cell copies a solution onto a student file**. Once Steps 0–9 and your earlier Article 2 exercises are finished, set `REFERENCE = False`, restart the kernel, and Run All. A passing reference run is not a grade for your unfinished exercises.

In [ ]:
from pathlib import Path
import importlib
from importlib.metadata import version
import subprocess
import sys
import shutil
import shlex

PART3 = Path.cwd().resolve()
assert PART3.name == "part3" and (PART3 / "final_check.py").is_file(), (
    "Start this kernel in <tutorial>/notebooks/newton/part3."
)
ROOT = PART3.parent.parent
REFERENCE = True
ROBOTS = ("so101", "rebot")
SCRIPT = PART3 / ("solutions/so101_newton_solution.py" if REFERENCE else "so101_newton.py")
print("Python:", sys.executable)
print("Mode:", "reference solutions" if REFERENCE else "your exercise files")
print("Newton entry point:", SCRIPT.relative_to(PART3))
for package in ("newton", "warp-lang", "mujoco", "mujoco-warp", "newton-usd-schemas", "usd-core"):
    print(f"{package}: {version(package)}")
assert version("newton") == "1.6.0", "Select the Article 3 kernel."


def run_python(script, *args, timeout=900):
    script = Path(script).resolve()
    command = [sys.executable, str(script), *map(str, args)]
    print("Running:", shlex.join(command))
    try:
        result = subprocess.run(command, cwd=script.parent, capture_output=True,
                                text=True, timeout=timeout, check=True)
    except subprocess.CalledProcessError as exc:
        print(exc.stdout or "")
        print(exc.stderr or "")
        raise
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result

## Overview of the files

| File | What you do with it |
|---|---|
| `newton_scene.py` | Complete `TODO Step 0` through `TODO Step 5` in your editor |
| `so101_newton.py` | Complete `TODO Step 6` through `TODO Step 9`; the entry-point name is historical and supports both robots |
| `solutions/newton_scene_solution.py` | Finished builder, joint mapping and target update helpers |
| `solutions/so101_newton_solution.py` | Finished `Example`, directly executable from `solutions/` |
| `solutions/step_00_imports.py` … `solutions/step_09_cuda_graph.py` | Focused references, intended to be read in the context of each step |
| `robots.py` | Robot-specific home pose, gripper channels, gains, force limits, scene layout and contact budgets |
| `pick_place_common.py`, `utils.py` | Reused IK/controller, generated scene and asset download helpers |
| `final_check.py` | Shared task thresholds and student/reference check selection |

The first asset use may download the pinned Menagerie model and populate `.generated/`. A network/import failure is not an expected exercise error.

Create backups **only if they do not already exist**. Re-running this cell must not replace an earlier baseline or alter your current solution.

In [ ]:
for name in ("newton_scene.py", "so101_newton.py"):
    source = PART3 / name
    backup = source.with_name(source.stem + "_original.py")
    if not backup.exists():
        shutil.copy2(source, backup)
        print("Created backup:", backup.name)
    else:
        print("Kept existing backup:", backup.name)

## Baseline: recognize the intentional failure

The untouched builder ends with a deliberate `NotImplementedError` naming Steps 1–5. Check that exact error before editing. You do **not** accept an arbitrary nonzero exit: a missing package, broken asset download, syntax error or unrelated exception must still stop the notebook.

After you remove the sentinel while completing the builder, this baseline check no longer applies. The later build and rollout assertions still have to pass. Reference mode does not turn a broken student script into a finished one.

In [ ]:
sentinel = 'raise NotImplementedError("Complete TODO Steps 1-5 in build_newton_model()")'
student_source = (PART3 / "newton_scene.py").read_text()
if sentinel in student_source:
    try:
        subprocess.run(
            [sys.executable, "-c", "from newton_scene import build_newton_model; build_newton_model()"],
            cwd=PART3, capture_output=True, text=True, timeout=180, check=True,
        )
    except subprocess.CalledProcessError as exc:
        final_error = (exc.stderr or "").strip().splitlines()[-1:]
        expected = "NotImplementedError: Complete TODO Steps 1-5 in build_newton_model()"
        if final_error != [expected]:
            print(exc.stdout or "")
            print(exc.stderr or "")
            raise  # An unrelated failure is NOT an expected baseline.
        print("EXPECTED incomplete-exercise error:", expected)
    else:
        raise AssertionError("The baseline sentinel exists, but the incomplete builder unexpectedly succeeded.")
else:
    print("Baseline sentinel removed: verify your edited builder and full task below.")

## Build the scene: `newton_scene.py`

### Step 0. Import Newton and choose the target layout

At `TODO Step 0`, import the Newton package, target-mode enum and MuJoCo solver. Newton 1.6.0 already defaults to coordinate-layout targets. Keep the assignment **before any builder or model is created** to document the policy; it is no longer an opt-in requirement:

```python
import newton
from newton import JointTargetMode
from newton.solvers import SolverMuJoCo

newton.use_coord_layout_targets = True
```

You still need `mujoco`: your controller uses a MuJoCo model for CPU-side kinematics. That does not mean MuJoCo is secretly stepping the scene outside Newton. In reference mode, the next cell imports the completed builder module without importing the incomplete student builder.

In [ ]:
import numpy as np
import warp as wp
import newton
from newton import JointTargetMode
from newton.solvers import SolverMuJoCo
from robots import get_robot

wp.init()
newton.use_coord_layout_targets = True
scene_module = importlib.import_module(
    "solutions.newton_scene_solution" if REFERENCE else "newton_scene"
)
print("Scene module:", Path(scene_module.__file__).relative_to(PART3))

### Step 1. Create the builder and register solver attributes

At `TODO Step 1`, create the builder, configure the detection gap and register MuJoCo-specific attributes **before importing**:

```python
builder = newton.ModelBuilder()
builder.default_shape_cfg.gap = 0.0
SolverMuJoCo.register_custom_attributes(builder)
```

The 1.6.0 `add_mjcf` importer registers MuJoCo custom attributes automatically. This explicit call documents the solver-specific authoring contract and is also used before the later USD import. Registration alone is not a USD schema resolver. The explicit zero gap preserves the Article 2 MJCF detection configuration and keeps task shapes from inheriting a different detection range. This is a task-specific parity choice, not a universal tuning recommendation.

The [v1.6.0 MuJoCo guide](https://github.com/newton-physics/newton/blob/v1.6.0/docs/solvers/mujoco.rst) documents registration and the distinction between contact margin and detection gap; [the MJCF importer](https://github.com/newton-physics/newton/blob/v1.6.0/newton/_src/sim/builder.py#L4196-L4235) documents automatic registration. Compare `solutions/step_01_builder.py`.


### Step 2. Import the robot, not a duplicate task scene

At `TODO Step 2`, import `robot_xml`, the same underlying robot model used in Article 2:

```python
builder.add_mjcf(
    str(robot_xml),
    ignore_names=["floor"],
    collapse_fixed_joints=True,
)
```

You skip the robot's floor because Step 5 adds the Newton ground plane. You merge fixed links to reduce the represented articulation; do not assume the resulting body indices match the MJCF indices. The table and cubes are added separately in Step 4, not imported twice.

`add_urdf` and `add_usd` are sibling importers, but changing file format also requires checking units, frames, collision geometry and importer support. Here you keep MJCF fixed so you can isolate the **API** migration. the clean-table notebook makes the asset change explicitly: official structured robot USD for dynamics, pinned in `newton_assets.py`, with the existing MJCF model retained for controller IK and reBot robot-only gravity feedforward.


### Step 3. Seed named joints and configure position servos

Do not hard-code “the first six entries.” The SO-101 has **six control channels** including its gripper; reBot has **eight**, including two gripper channels. Each free cube adds coordinates and DOFs that are **not** actuators.

Match each MJCF actuator's joint name to a Newton joint label with `robot_joint_indices`. Then use the appropriate layout:

| Quantity | Offset you use |
|---|---|
| Builder initial joint position | `builder.joint_q_start[joint]` |
| Builder position target, with coordinate layout enabled | The same coordinate offset in `builder.joint_target_q` |
| Gains, damping, effort limits and target mode | `builder.joint_qd_start[joint]` |
| Finalized model's runtime position-target index | `model.joint_target_q_start[joint]` |

At `TODO Step 3`, seed both actual joint coordinates and targets from `spec.home_ctrl` in actuator order. Setting the pose but leaving stale targets can make the arm lurch on the first physics frame. Set `JointTargetMode.POSITION`, stiffness and damping using the profile, not a single universal arm constant.

Preserve gripper force limits. For an arm-only effort boost, consider both Newton's joint limit and the imported `mujoco:actuator_forcerange`; raising only one can leave the actuator capped. The reference applies the Article 2 policy and preserves reBot's heterogeneous imported force limits. Read `solutions/step_03_joint_targets.py` alongside the complete helper; copying SO-101-specific slices is not a migration for both robots.

### Step 4. Recreate the static table and dynamic cubes

Attach the table to `body=-1`, meaning the world. Each cube gets a dynamic body and a box shape. Use the selected profile's positions and half-extents through `task`: scene resolution updates its active-robot aliases, so prefer `import pick_place_common as task` over caching one layout before selecting the robot.

The MJCF cube half-extent is `0.022 m` and its mass is **`0.08 kg`**. Preserve both, not just its color. For a solid cube with side length `s`, each principal inertia is `mass * s**2 / 6`. This fragment belongs inside the `(label, pos, color)` loop in `solutions/step_04_props.py`; `label` and `pos` come from that loop:

```python
cube_mass = 0.08
cube_inertia = wp.mat33(np.eye(3) * cube_mass * (2.0 * task.CUBE_HALF)**2 / 6.0)
body = builder.add_body(
    xform=wp.transform(wp.vec3(*pos), wp.quat_identity()),
    label=label, mass=cube_mass, inertia=cube_inertia, lock_inertia=True,
)
```

Use zero shape density when mass/inertia were set on the body, so you do not add mass twice. A zero-density shape does **not** make the cube static: the world attachment makes the table static. Preserve the task's friction and contact configuration as well as the dimensions. See `solutions/step_04_props.py`.

### Step 5. Add the ground, finalize, and check both builds

At `TODO Step 5`, finish the model:

```python
builder.add_ground_plane()
return builder.finalize()
```

Finalization fixes the allocated topology and creates device arrays; it does not make every model parameter read-only. If you later change parameters that a solver caches, use the supported `notify_model_changed` path. Here you configure them before constructing the solver.

The next checkpoint builds each robot on **CPU**, resolves cube bodies by label, checks their masses, maps all control targets and compares them with the home pose. These are model checks, **not** proof that the manipulation succeeds.

In [ ]:
models = {}
for robot in ROBOTS:
    spec = get_robot(robot)
    with wp.ScopedDevice("cpu"):
        model = scene_module.build_newton_model(spec)
        ik_model, _, _ = scene_module.build_ik_model(spec)
        joints = scene_module.robot_joint_indices(model, ik_model)
        target_indices = model.joint_target_q_start.numpy()[joints]
        control = model.control()
        state = model.state()
        newton.eval_fk(model, model.joint_q, model.joint_qd, state)
    red = scene_module.body_index(model, "red_cube")
    blue = scene_module.body_index(model, "blue_cube")
    expected_controls = {"so101": 6, "rebot": 8}[robot]
    assert len(joints) == len(spec.home_ctrl) == ik_model.nu == expected_controls
    assert len(set(map(int, target_indices))) == expected_controls
    assert np.allclose(model.body_mass.numpy()[[red, blue]], 0.08)
    assert np.allclose(control.joint_target_q.numpy()[target_indices], spec.home_ctrl)
    assert np.isfinite(state.body_q.numpy()).all()
    models[robot] = model
    print(robot, "bodies=", model.body_count, "DOFs=", model.joint_dof_count,
          "control channels=", expected_controls, "target indices=", target_indices.tolist())
assert set(models) == set(ROBOTS)

## Drive the simulation: `so101_newton.py`

### Step 6. Allocate state, controls and the selected contact path

At `TODO Step 6`, replace the placeholder buffers with two states and a control object, then run forward kinematics:

```python
self.state_0 = self.model.state()
self.state_1 = self.model.state()
self.control = self.model.control()
newton.eval_fk(self.model, self.model.joint_q, self.model.joint_qd, self.state_0)
```

Newton has split the roles that `MjData` combined. You keep the current state separate from next-state output; contact storage depends on the detector selected in Step 7.

Your CPU kinematic model still supports `PickPlaceController`. After mapping its actuator joints, update **only** `control.joint_target_q` at `self.model.joint_target_q_start.numpy()[joints]`. The reference `set_joint_targets(control, ctrl, target_indices)` checks the vector shape, preserves unrelated target entries, and copies into the existing device array. This transparent host-based controller is for parity, not a device-resident learning benchmark.

### Step 7. Construct SolverMuJoCo and choose the contact detector

At `TODO Step 7`, construct `newton.solvers.SolverMuJoCo`. You should recognize `nconmax` and `njmax`: these are the MuJoCo Warp capacity budgets surfaced through Newton. Use the profile's contact capacity (`nconmax=self.spec.nconmax`) and at least 1024 constraint rows (`njmax=max(self.spec.njmax, 1024)`). A capacity overflow invalidates the rollout. Iteration and line-search limits are separate convergence diagnostics and should not be confused with storage overflow.

| Execution choice | Underlying dynamics | Contact choice in this lesson |
|---|---|---|
| `--device cpu` | `use_mujoco_cpu=True`: native MuJoCo-C | MuJoCo detection; the native path does not consume Newton-generated contacts in 1.6.0 |
| CUDA device | `use_mujoco_cpu=False`: MuJoCo Warp | Newton or MuJoCo detection; validate the requested path explicitly |

The reference keeps the task's constraint/integrator settings (`solver="newton"`, `integrator="implicitfast"`, elliptic friction cone and the capacities described above). The `"newton"` string here names MuJoCo's constraint-solving algorithm, not a second Newton physics engine.

For Newton detection, create a `newton.CollisionPipeline(self.model)` and allocate `self.contacts = pipeline.contacts()`. For native MuJoCo detection, the solver finds contacts internally and the external contact argument can be `None`. Setting the Warp device to CPU alone is not the same as requesting native MuJoCo-C.

Inspect the actual entry point's help below before selecting a GPU detector. The startup log reports the chosen dynamics backend and contact path; a CPU native result cannot establish correctness of the CUDA Newton-contact path.

In [ ]:
cli_help = run_python(SCRIPT, "--help")
assert "--device" in cli_help.stdout and "--robot" in cli_help.stdout
assert "--test" in cli_help.stdout

### Step 8. Write the substep loop and preserve the clock

The controller runs at **50 Hz** with **10 physics substeps** per frame. Both the Newton timestep and the Article 2 CPU baseline must agree with `frame_dt / sim_substeps`, rather than inheriting the robot asset's timestep:

```python
self.fps = 50
self.frame_dt = 1.0 / self.fps
self.sim_substeps = 10
self.sim_dt = self.frame_dt / self.sim_substeps
```

Compute one control vector per frame and hold it across the substeps. When Newton detects contacts, run the pipeline on the current input state at **each substep**, not once for the whole controller frame:

```python
for _ in range(self.sim_substeps):
    self.state_0.clear_forces()
    if self.collision_pipeline is not None:
        self.collision_pipeline.collide(self.state_0, self.contacts)
        wp.launch(_record_contact_peak, dim=1,
                  inputs=[self.contacts.rigid_contact_count, self.contact_peak],
                  device=self.model.device)
    self.solver.step(self.state_0, self.state_1, self.control, self.contacts, self.sim_dt)
    self.state_0, self.state_1 = self.state_1, self.state_0
```

The exercise already defines `_record_contact_peak` and allocates `self.contact_peak`. This kernel retains the largest contact count across every substep, including inside a CUDA graph, so `--test` can detect lost contacts even when the last frame has none.

The swap allocates nothing. Forget it and you keep reading old state; omit the physics call and you are only moving a kinematic model. Once the rollout starts, never reposition the cubes to satisfy the final test. Contact, friction and the gradual gripper command must do the work.

| Backend | Inner step |
|---|---|
| MuJoCo CPU | `mujoco.mj_step(model, data)` |
| MuJoCo Warp | `mjw.step(model, data)` |
| Newton | `solver.step(state_0, state_1, control, contacts, dt)` |

In [ ]:
fps, sim_substeps = 50, 10
frame_dt = 1.0 / fps
sim_dt = frame_dt / sim_substeps
print("Controller dt:", frame_dt, "s; physics dt:", sim_dt, "s")
assert np.isclose(sim_dt, 0.002)
assert np.isclose(600 * sim_substeps * sim_dt, 600 * frame_dt)

### Step 9. Capture a supported CUDA frame — not the host controller

At `TODO Step 9`, leave `self.graph = None` on the native CPU path. On CUDA you can capture the supported simulation work once all buffers and the solver exist:

```python
self.graph = None
if self.model.device.is_cuda:
    with wp.ScopedCapture() as capture:
        self.simulate()
    self.graph = capture.graph
```

The host controller writes new values into the same control array before each replay. `wp.capture_launch` replays recorded work; it does not rerun Python waypoint logic. Python reference swaps occur while recording, so the even substep count keeps the frame's buffer roles consistent between replays. Replacing buffers or rebuilding topology requires a new capture; validate initialization and replay behavior rather than assuming capture is an extra elapsed physics frame.

This is **not** a claim that Warp graphs are categorically CUDA-only: Warp also has CPU graph capabilities. Here you deliberately leave the native MuJoCo-C stepping path uncaptured. Nor does capture enable autodiff for `SolverMuJoCo`, which is not differentiable in this release.

Run uncaptured/headless CPU correctness first. A CUDA graph or a high FPS number cannot repair incorrect targets or a failed grasp.

## Verify the task: CPU baseline, then Newton, for both robots

Now run the selected Article 2 CPU implementation and the Newton migration. These are **current-environment reruns** of Article 2 code, not a reproduction of the older Article 2 environment. `REFERENCE = True` chooses both reference entry points; `False` chooses both sets of student exercises.

Each run must finish normally and print finite final metrics. You require horizontal alignment `xy_err <= 0.015 m` and center-height separation `0.035 <= dz <= 0.055 m`. These describe a stack of the task's cubes, not merely a cube somewhere above the table. Newton's `--test` additionally checks its final-state/controller assertions.

The loop below checks every requested robot/backend pair. It does not count a missing report as success, hide a traceback, or label an unfinished exercise a GPU skip.

In [ ]:
from final_check import checks_for, parse_stack, MAX_XY_ERROR_M, MIN_DZ_M, MAX_DZ_M

measurements = []
for robot in ROBOTS:
    cpu_checks = [check for check in checks_for(robot, solutions=REFERENCE) if not check.requires_cuda]
    for check in cpu_checks:
        result = run_python(check.script, *check.args, timeout=check.timeout)
        metrics = parse_stack(result.stdout)
        assert metrics is not None, f"No finite final stack report: {check.name}"
        xy_err, dz = metrics
        assert 0.0 <= xy_err <= MAX_XY_ERROR_M, (check.name, xy_err)
        assert MIN_DZ_M <= dz <= MAX_DZ_M, (check.name, dz)
        measurements.append({"robot": robot, "script": check.script.name, "xy_err": xy_err, "dz": dz})
expected_pairs = {
    (robot, check.script.name)
    for robot in ROBOTS
    for check in checks_for(robot, solutions=REFERENCE)
    if not check.requires_cuda
}
assert {(row["robot"], row["script"]) for row in measurements} == expected_pairs
assert len(measurements) == len(expected_pairs)
for row in measurements:
    print(row)
print("PASS: all requested CPU baseline and Newton stack checks. CUDA remains a separate check.")

## Optional CUDA comparison: choose and verify the detector

You can enable the following cell on a CUDA machine **after** the CPU checks pass. The cell first asserts that CUDA is available; a requested but unavailable GPU run is not silently replaced by CPU physics. It runs both detector choices exposed by the entry point's help, with `--test`, and checks each final stack report.

The default `RUN_GPU = False` keeps Run All portable. It means **not verified**, even on a machine that happens to have CUDA. Contact generation can differ across native MuJoCo and Newton pipelines; support for SDF or hydroelastic contact does not automatically configure those representations for imported meshes. Investigate discrepancies instead of relaxing the task criteria.

These are one-world correctness runs, not a many-world throughput study. If you later benchmark, warm up and synchronize, remove host transfers from the timed loop, and report the device, world count and measured world-steps per second.

In [ ]:
RUN_GPU = False
if RUN_GPU:
    assert wp.is_cuda_available(), "Requested CUDA comparison, but Warp found no CUDA device."
    assert "--use-mujoco-contacts" in cli_help.stdout, "Inspect the contact CLI."
    detector_args = {"Newton": ["--no-use-mujoco-contacts"],
                     "MuJoCo": ["--use-mujoco-contacts"]}
    gpu_measurements = []
    for robot in ROBOTS:
        for detector, flags in detector_args.items():
            result = run_python(SCRIPT, "--robot", robot, "--device", "cuda:0", "--viewer", "null",
                                "--num-frames", "600", "--test", *flags)
            metrics = parse_stack(result.stdout)
            assert metrics is not None, (robot, detector, "missing report")
            xy_err, dz = metrics
            assert 0.0 <= xy_err <= MAX_XY_ERROR_M and MIN_DZ_M <= dz <= MAX_DZ_M
            gpu_measurements.append((robot, detector, xy_err, dz))
    assert len(gpu_measurements) == len(ROBOTS) * len(detector_args)
    print(gpu_measurements)
else:
    print("SKIP: optional CUDA/contact-pipeline comparison not requested; remains unverified.")

## Watch it in Newton's viewer

The `null` viewer above avoids display requirements. To watch the same selected program, run one of the following commands in a local terminal. The printed command uses this kernel's interpreter, not a `python` found elsewhere on PATH.

Newton's viewer renders Newton `State` objects; you are not mirroring into `mujoco.viewer`. If a desktop/OpenGL issue prevents a window, the headless task remains the debugging path. Do not use an interactive window as a substitute for `--test`.

In [ ]:
for robot in ROBOTS:
    print(shlex.join([sys.executable, str(SCRIPT), "--robot", robot, "--device", "cpu",
                      "--viewer", "gl", "--num-frames", "600", "--test"]))

## Compare your files without modifying them

Use a normal text diff to compare your implementation with the references. Import shims differ because references run directly from `solutions/`; focus on scene semantics, target indices and the physics loop. A textual difference is information, not a swallowed execution error.

The next cell prints a review diff in student mode. It never applies the diff or copies over your work.

In [ ]:
if not REFERENCE:
    import difflib
    for student, reference in (
        ("newton_scene.py", "solutions/newton_scene_solution.py"),
        ("so101_newton.py", "solutions/so101_newton_solution.py"),
    ):
        diff = difflib.unified_diff(
            (PART3 / student).read_text().splitlines(),
            (PART3 / reference).read_text().splitlines(),
            fromfile=student, tofile=reference, lineterm="",
        )
        print("\n".join(diff))
else:
    print("You ran references. Complete the external exercises, set REFERENCE=False, restart and rerun.")

## Troubleshooting

| Symptom | What you should check |
|---|---|
| A traceback appears in the baseline cell | Only the exact named `NotImplementedError` is expected; package, network and syntax errors require fixing. |
| One robot builds but the other fails | Use profile data and actuator-to-joint name matching; do not assume six controls or one gripper channel. |
| The arm lurches on the first frame | Initial `joint_q` and `joint_target_q` should both match the selected home pose. |
| Targets move a free cube or the wrong joint | Distinguish coordinate, DOF and runtime target offsets; use `joint_target_q_start`. |
| The robot cannot lift the cube | Preserve the 0.08 kg mass, gripper limits and arm-only force policy, including imported actuator limits. |
| Controller timing differs from Article 2 | Derive 0.002 s physics from 50 Hz and 10 substeps in both environments. |
| Native CPU ignores your external Newton contacts | The native MuJoCo-C path requires its own detector; test Newton contacts on the supported MJWarp path separately. |
| The cube moves but never stacks | Check actual substep stepping, swaps, target updates, contact configuration and the gradual frictional grasp. Never teleport the payload or weaken the thresholds. |
| Missing final metrics or nonzero exit | This is a failure, even if the viewer looked convincing. Read the complete subprocess output. |
| GPU comparison says SKIP | It was not run. Enable it on supported CUDA hardware before claiming that detector/backend was verified. |

## Recap

Before moving on, make sure you can answer:

- What stayed fixed when you moved the task to Newton?
- Why do import-time custom attributes and robot-specific force limits matter?
- Why do both robots need name-based target mapping?
- What is the difference between native CPU, Warp CPU and CUDA stepping?
- Why are contact detection and solver integration separate choices?
- What did your assertions establish, and which optional checks remain unrun?

## Next

Continue to **[Clean the table](03__clean_the_table.ipynb)**, where Newton's wider model becomes necessary: a rigid robot and clutter interact with a deformable shirt and cable through explicit solver coupling. That advanced task has different outcome criteria from stacking.

Then open **[Final check](04__final_check.ipynb)** to assert the selected student/reference implementations for both robots, including the advanced task. A reference pass, a student pass and a GPU skip are three different results.